# **Comparison Study**

In [ ]:
# ===================== REIMPLEMENTATION (FULL) =====================

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.feature_selection import SelectFromModel

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

from sklearn.base import clone
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD TRAIN DATA
# ============================================================

train_path = "/content/drive/MyDrive/Colab Notebooks/My Thesis on ASD Education/ASD_Datasets/ToddlerGroup_MERGED_train.csv"
test_path  = "/content/drive/MyDrive/Colab Notebooks/My Thesis on ASD Education/ASD_Datasets/ToddlerGroup_MERGED_test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

# Clean placeholders
placeholders = ['?', '.', 'NA', '', '@', 'NONE']
df_train.replace(placeholders, np.nan, inplace=True)
df_test.replace(placeholders, np.nan, inplace=True)

# Split features/target
drop_cols = ["Case_No", "Class_ASD", "Score", "Teaching_Strategy"]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train["Teaching_Strategy"]

X_test = df_test.drop(columns=drop_cols)
y_test = df_test["Teaching_Strategy"]

# ============================================================
# 2. PREPROCESSING (CONTROLLED - SAME FOR ALL MODELS)
# ============================================================

numeric_cols = [f"A{i}" for i in range(1, 11)] + ["Age"]
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), numeric_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

# Encode target
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train.astype(str))
y_test_enc  = le.transform(y_test.astype(str))
n_classes = len(le.classes_)

# ============================================================
# 3. DEFINE MODELS
# ============================================================

# --- Proposed Model (Your RFC - FIXED PARAMS) ---
proposed_pipe = Pipeline([
    ("preproc", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=3,
        min_samples_split=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

# --- Hajjej Reimplementation ---
hajjej_pipe = Pipeline([
    ("preproc", preprocessor),
    ("feature_selection", SelectFromModel(
        ExtraTreesClassifier(n_estimators=100, random_state=42),
        threshold="mean"
    )),
    ("clf", VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=20, random_state=25)),
            ('xgb', XGBClassifier(tree_method="hist", cat_smooth=10, cat_l2=1.0, random_state=25))
        ],
        voting='soft'
    ))
])

# --- Zoana Baseline ---
zoana_pipe = Pipeline([
    ("preproc", preprocessor),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

models = {
    "Proposed": proposed_pipe,
    "Hajjej": hajjej_pipe,
    "Zoana": zoana_pipe
}

# ============================================================
# 4. 5-FOLD CROSS VALIDATION (ROBUST COMPARISON)
# ============================================================

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {
    name: {"acc":[], "prec":[], "rec":[], "f1_w":[], "f1_m":[], "auc":[]}
    for name in models
}

print("Running 5-Fold Controlled CV...\n")

for fold, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train, y_train_enc), 1):

    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_enc[tr_idx], y_train_enc[val_idx]

    y_val_bin = label_binarize(y_val, classes=range(n_classes))

    for name, pipe in models.items():

        model = clone(pipe)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)

        results[name]["acc"].append(accuracy_score(y_val, y_pred))
        results[name]["prec"].append(precision_score(y_val, y_pred, average='weighted'))
        results[name]["rec"].append(recall_score(y_val, y_pred, average='weighted'))
        results[name]["f1_w"].append(f1_score(y_val, y_pred, average='weighted'))
        results[name]["f1_m"].append(f1_score(y_val, y_pred, average='macro'))
        results[name]["auc"].append(
            roc_auc_score(y_val_bin, y_prob, average='weighted', multi_class='ovr')
        )

    print(f"Fold {fold} done")

# ============================================================
# 5. CV RESULTS
# ============================================================

print("\n" + "="*100)
print("CROSS-VALIDATION RESULTS")
print("="*100)

print(f"{'MODEL':10s} | {'ACC':18s} | {'Precision':18s} | {'Recall':18s} | {'F1_w':18s} | {'F1_macro':18s} | {'ROC_AUC':18s}")
print("-"*100)

for name in models:
    print(f"{name:10s} | "
          f"{np.mean(results[name]['acc']):.4f}±{np.std(results[name]['acc']):.4f} | "
          f"{np.mean(results[name]['prec']):.4f}±{np.std(results[name]['prec']):.4f} | "
          f"{np.mean(results[name]['rec']):.4f}±{np.std(results[name]['rec']):.4f} | "
          f"{np.mean(results[name]['f1_w']):.4f}±{np.std(results[name]['f1_w']):.4f} | "
          f"{np.mean(results[name]['f1_m']):.4f}±{np.std(results[name]['f1_m']):.4f} | "
          f"{np.mean(results[name]['auc']):.4f}±{np.std(results[name]['auc']):.4f}")

print("="*100)

# ============================================================
# 6. FINAL TEST EVALUATION (CRITICAL FOR PAPER)
# ============================================================
print("=============Toddler=========================")
print("\n" + "="*100)
print("FINAL TEST SET RESULTS")
print("="*100)

print(f"{'MODEL':10s} | {'ACC':10s} | {'Precision':10s} | {'Recall':10s} | {'F1_w':10s} | {'F1_macro':10s} | {'ROC_AUC':10s}")
print("-"*100)

y_test_bin = label_binarize(y_test_enc, classes=range(n_classes))

for name, pipe in models.items():

    model = clone(pipe)
    model.fit(X_train, y_train_enc)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    acc = accuracy_score(y_test_enc, y_pred)
    prec = precision_score(y_test_enc, y_pred, average='weighted')
    rec = recall_score(y_test_enc, y_pred, average='weighted')
    f1_w = f1_score(y_test_enc, y_pred, average='weighted')
    f1_m = f1_score(y_test_enc, y_pred, average='macro')
    auc = roc_auc_score(y_test_bin, y_prob, average='weighted', multi_class='ovr')

    print(f"{name:10s} | {acc:.4f}   | {prec:.4f}   | {rec:.4f}   | {f1_w:.4f}   | {f1_m:.4f}   | {auc:.4f}")

print("="*100)


import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.feature_selection import SelectFromModel

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

from sklearn.base import clone
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1. LOAD TRAIN DATA
# ============================================================

train_path = "/content/drive/MyDrive/Colab Notebooks/My Thesis on ASD Education/ASD_Datasets/ChildGroup_MERGED_train.csv"
test_path  = "/content/drive/MyDrive/Colab Notebooks/My Thesis on ASD Education/ASD_Datasets/ChildGroup_MERGED_test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

# Clean placeholders
placeholders = ['?', '.', 'NA', '', '@', 'NONE']
df_train.replace(placeholders, np.nan, inplace=True)
df_test.replace(placeholders, np.nan, inplace=True)

# Split features/target
drop_cols = ["Case_No", "Class_ASD", "Score", "Teaching_Strategy"]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train["Teaching_Strategy"]

X_test = df_test.drop(columns=drop_cols)
y_test = df_test["Teaching_Strategy"]

# ============================================================
# 2. PREPROCESSING (CONTROLLED - SAME FOR ALL MODELS)
# ============================================================

numeric_cols = [f"A{i}" for i in range(1, 11)] + ["Age"]
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ]), numeric_cols),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop="first", handle_unknown="ignore",min_frequency=0.05 , sparse_output=False))
    ]), categorical_cols)
])

# Encode target
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train.astype(str))
y_test_enc  = le.transform(y_test.astype(str))
n_classes = len(le.classes_)

# ============================================================
# 3. DEFINE MODELS
# ============================================================

# --- Proposed Model (Your RFC - FIXED PARAMS) ---
proposed_pipe = Pipeline([
    ("preproc", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_leaf=4,
        min_samples_split=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

# --- Hajjej Reimplementation ---
hajjej_pipe = Pipeline([
    ("preproc", preprocessor),
    ("feature_selection", SelectFromModel(
        ExtraTreesClassifier(n_estimators=100, random_state=42),
        threshold="mean"
    )),
    ("clf", VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=20, random_state=25)),
            ('xgb', XGBClassifier(tree_method="hist", cat_smooth=10, cat_l2=1.0, random_state=25))
        ],
        voting='soft'
    ))
])

# --- Zoana Baseline ---
zoana_pipe = Pipeline([
    ("preproc", preprocessor),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

models = {
    "Proposed": proposed_pipe,
    "Hajjej": hajjej_pipe,
    "Zoana": zoana_pipe
}

# ============================================================
# 4. 5-FOLD CROSS VALIDATION (ROBUST COMPARISON)
# ============================================================

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {
    name: {"acc":[], "prec":[], "rec":[], "f1_w":[], "f1_m":[], "auc":[]}
    for name in models
}

print("Running 5-Fold Controlled CV...\n")

for fold, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train, y_train_enc), 1):

    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_enc[tr_idx], y_train_enc[val_idx]

    y_val_bin = label_binarize(y_val, classes=range(n_classes))

    for name, pipe in models.items():

        model = clone(pipe)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)

        results[name]["acc"].append(accuracy_score(y_val, y_pred))
        results[name]["prec"].append(precision_score(y_val, y_pred, average='weighted'))
        results[name]["rec"].append(recall_score(y_val, y_pred, average='weighted'))
        results[name]["f1_w"].append(f1_score(y_val, y_pred, average='weighted'))
        results[name]["f1_m"].append(f1_score(y_val, y_pred, average='macro'))
        results[name]["auc"].append(
            roc_auc_score(y_val_bin, y_prob, average='weighted', multi_class='ovr')
        )

    print(f"Fold {fold} done")

# ============================================================
# 5. CV RESULTS
# ============================================================
print("========================Child==================================")
print("\n" + "="*100)
print("CROSS-VALIDATION RESULTS")
print("="*100)

print(f"{'MODEL':10s} | {'ACC':18s} | {'Precision':18s} | {'Recall':18s} | {'F1_w':18s} | {'F1_macro':18s} | {'ROC_AUC':18s}")
print("-"*100)

for name in models:
    print(f"{name:10s} | "
          f"{np.mean(results[name]['acc']):.4f}±{np.std(results[name]['acc']):.4f} | "
          f"{np.mean(results[name]['prec']):.4f}±{np.std(results[name]['prec']):.4f} | "
          f"{np.mean(results[name]['rec']):.4f}±{np.std(results[name]['rec']):.4f} | "
          f"{np.mean(results[name]['f1_w']):.4f}±{np.std(results[name]['f1_w']):.4f} | "
          f"{np.mean(results[name]['f1_m']):.4f}±{np.std(results[name]['f1_m']):.4f} | "
          f"{np.mean(results[name]['auc']):.4f}±{np.std(results[name]['auc']):.4f}")

print("="*100)

# ============================================================
# 6. FINAL TEST EVALUATION (CRITICAL FOR PAPER)
# ============================================================

print("\n" + "="*100)
print("FINAL TEST SET RESULTS")
print("="*100)

print(f"{'MODEL':10s} | {'ACC':10s} | {'Precision':10s} | {'Recall':10s} | {'F1_w':10s} | {'F1_macro':10s} | {'ROC_AUC':10s}")
print("-"*100)

y_test_bin = label_binarize(y_test_enc, classes=range(n_classes))

for name, pipe in models.items():

    model = clone(pipe)
    model.fit(X_train, y_train_enc)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    acc = accuracy_score(y_test_enc, y_pred)
    prec = precision_score(y_test_enc, y_pred, average='weighted')
    rec = recall_score(y_test_enc, y_pred, average='weighted')
    f1_w = f1_score(y_test_enc, y_pred, average='weighted')
    f1_m = f1_score(y_test_enc, y_pred, average='macro')
    auc = roc_auc_score(y_test_bin, y_prob, average='weighted', multi_class='ovr')

    print(f"{name:10s} | {acc:.4f}   | {prec:.4f}   | {rec:.4f}   | {f1_w:.4f}   | {f1_m:.4f}   | {auc:.4f}")

print("="*100)

Running 5-Fold Controlled CV...

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

CROSS-VALIDATION RESULTS
MODEL      | ACC                | Precision          | Recall             | F1_w               | F1_macro           | ROC_AUC           
----------------------------------------------------------------------------------------------------
Proposed   | 0.9702±0.0097 | 0.9714±0.0092 | 0.9702±0.0097 | 0.9674±0.0112 | 0.9151±0.0295 | 0.9993±0.0003
Hajjej     | 0.9839±0.0077 | 0.9848±0.0075 | 0.9839±0.0077 | 0.9840±0.0076 | 0.9624±0.0213 | 0.9996±0.0004
Zoana      | 0.9919±0.0088 | 0.9924±0.0087 | 0.9919±0.0088 | 0.9918±0.0092 | 0.9846±0.0174 | 0.9997±0.0005
=============Toddler=========================

FINAL TEST SET RESULTS
MODEL      | ACC        | Precision  | Recall     | F1_w       | F1_macro   | ROC_AUC   
----------------------------------------------------------------------------------------------------
Proposed   | 0.9871   | 0.9876   | 0.9871   | 0.9870   | 0.983